Get Data from API using python requests 

In [0]:
import requests
import json
from datetime import datetime


url = "http://inceptezlabs.com/api.php"

headers = {
    "User-Agent": "Chrome/151.0.0.0 Safari/537.36"
}


resp = requests.get(url,headers=headers)
data=resp.json()

print(type(data))
ts = datetime.now().strftime("%Y%m%d%H%M%S")
output_path = f"/Volumes/lakehousecat1/deltadb/datalake/we48_lakeflow/apidata/posts_{ts}.json"

dbutils.fs.put(
                output_path,
                json.dumps(data),
                overwrite=True
            )

In [0]:

from pyspark.sql import functions as F

df_raw = (spark.readStream
        .format("cloudFiles")#.schema(user_schema)
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes",True)
        #.option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.schemaLocation","/Volumes/lakehousecat1/deltadb/datalake/we48_lakeflow/api_schema")
        .option("cloudFiles.maxFilesPerTrigger", 1)
        .load("/Volumes/lakehousecat1/deltadb/datalake/we48_lakeflow/apidata/"))


df_user = (
    df_raw
        .select(
            F.col("data.uid").alias("uid"),
            F.col("data.user.name").alias("user_name"),
            F.col("data.user.email").alias("user_email"),
            F.col("data.user.location").alias("user_location"),
            F.to_timestamp(
                F.col("data.user.registered")
            ).alias("user_registered_ts"),
            F.current_timestamp().alias("ingestion_ts")
        )
)


(df_user.writeStream
        .format("delta")
        .trigger(availableNow=True)
        .option(
            "checkpointLocation",
            "/Volumes/lakehousecat1/deltadb/datalake/we48_lakeflow/api_checkpoint"
        )
        .toTable("lakehousecat1.deltadb.tblizdetail"))

In [0]:
%sql

select * from lakehousecat1.deltadb.tblizdetail


DBSQL / Hive  - complex datatype 

Regular Datatype : int, string , timestamp , date etc..  (hold single value )

Complex Type : (hold multiple value  )

Array :
        ordered collection of same type of elements , index based 

map :
     un ordered collection of key value pairs , key based 

struct :
        record type , column contain a row , value in diffrent types 
  

complex (not required)


In [0]:
%sql



create table lakehousecat1.deltadb.tbl_collection_details
(
    userid int,
    name string,
    lang array<string>,
    marks map<string,int>,
    address struct<street:string,city:string,zip:int>
);

-- user_addrs array<struct<street:string,city:string,zip:int>>

describe formatted lakehousecat1.deltadb.tbl_collection_details;


insert into lakehousecat1.deltadb.tbl_collection_details values 
( 100,
 'John',
 array('english','hindi'),
 map('maths',90,'science',80),
 struct("123 street","chennai",600008)
),
( 101,
 'Mary',
 array('Tamil','hindi',"English"),
 map('maths',95,'science',85,"cs",80),
 struct("123 street","chennai",600008)
),
( 102,
 'Ravi',
 array('Telungu','hindi',"spanish"),
 map('maths',80,'science',70,"cs",75),
 struct("123 street","chennai",600008)
)
 ;

select * from lakehousecat1.deltadb.tbl_collection_details;

-- userid,name , pref_lang,maths,city,zip

select userid,name,lang[0] as pref_lang,marks['maths'] as maths_mark,address.city,address.zip from lakehousecat1.deltadb.tbl_collection_details;


